In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Krippendorff's Alpha 计算脚本
用于评估多个编码者之间的一致性
"""

import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def krippendorff_alpha(data, level_of_measurement='nominal'):
    """
    计算 Krippendorff's Alpha
    
    参数:
    data: numpy array, 形状为 (n_coders, n_items)
          每一行代表一个编码者，每一列代表一个样本
    level_of_measurement: str, 测量水平
          'nominal' - 名义水平（分类数据）
          'ordinal' - 有序水平
          'interval' - 区间水平
          'ratio' - 比率水平
    
    返回:
    alpha: float, Krippendorff's Alpha 值
    """
    
    # 转换为numpy数组
    data = np.array(data)
    
    # 获取所有唯一的类别（排除缺失值）
    categories = []
    for row in data:
        categories.extend([x for x in row if x is not None and pd.notna(x)])
    categories = sorted(set(categories))
    
    # 创建类别到索引的映射
    cat_to_idx = {cat: idx for idx, cat in enumerate(categories)}
    n_categories = len(categories)
    
    # 计算配对矩阵（coincidence matrix）
    coincidence_matrix = np.zeros((n_categories, n_categories))
    
    n_coders, n_items = data.shape
    
    # 对每个样本
    for item_idx in range(n_items):
        # 获取该样本的所有编码（排除缺失值）
        item_codes = []
        for coder_idx in range(n_coders):
            code = data[coder_idx, item_idx]
            if code is not None and pd.notna(code):
                item_codes.append(code)
        
        # 如果至少有2个编码者对该样本进行了编码
        m_u = len(item_codes)
        if m_u > 1:
            # 对每对编码进行配对
            for i, code_i in enumerate(item_codes):
                for j, code_j in enumerate(item_codes):
                    if i != j:
                        idx_i = cat_to_idx[code_i]
                        idx_j = cat_to_idx[code_j]
                        # 权重为 1/(m_u - 1)
                        coincidence_matrix[idx_i, idx_j] += 1.0 / (m_u - 1)
    
    # 计算边际和
    n_c = coincidence_matrix.sum(axis=1)  # 每个类别的总数
    n_total = n_c.sum()  # 总配对数
    
    # 计算观察到的不一致性 (observed disagreement)
    D_o = 0.0
    for c in range(n_categories):
        for k in range(n_categories):
            if c != k:  # 名义水平：只有不同类别才计入不一致
                D_o += coincidence_matrix[c, k]
    
    # 计算期望的不一致性 (expected disagreement)
    D_e = 0.0
    for c in range(n_categories):
        for k in range(n_categories):
            if c != k:  # 名义水平：只有不同类别才计入不一致
                D_e += n_c[c] * n_c[k]
    
    # 归一化
    if n_total > 0:
        D_o = D_o / n_total
        D_e = D_e / (n_total * (n_total - 1))
    
    # 计算 Alpha
    if D_e == 0:
        alpha = 1.0  # 完全一致
    else:
        alpha = 1.0 - (D_o / D_e)
    
    return alpha, coincidence_matrix, categories


def analyze_disagreements(data, categories):
    """
    分析不一致的案例
    """
    data = np.array(data)
    n_coders, n_items = data.shape
    
    disagreement_cases = []
    
    for item_idx in range(n_items):
        item_codes = []
        for coder_idx in range(n_coders):
            code = data[coder_idx, item_idx]
            if code is not None and pd.notna(code):
                item_codes.append(code)
        
        # 检查是否有不一致
        if len(set(item_codes)) > 1:
            disagreement_cases.append({
                'item_id': item_idx + 1,
                'codes': item_codes,
                'n_unique': len(set(item_codes))
            })
    
    return disagreement_cases


def main():
    # 读取数据
    print("=" * 80)
    print("Krippendorff's Alpha 一致性分析")
    print("=" * 80)
    print()
    
    # 读取CSV文件
    df = pd.read_csv('/Users/majun/Documents/research/论文撰写/ LLM LoRA-ship collision/论文/投稿/12.29修改/补实验/一致性检查实验/一致性检查-四位人员编码数据.csv')
    
    # 显示数据基本信息
    print(f"数据集包含 {len(df)} 个样本，{len(df.columns)-1} 个编码者")
    print(f"编码者: {', '.join(df.columns[1:].tolist())}")
    print()
    
    # 准备数据矩阵（转置，使得每行是一个编码者）
    coder_columns = ['coder_A', 'coder_B', 'coder_C', 'coder_D']
    data_matrix = df[coder_columns].values.T  # 转置为 (n_coders, n_items)
    
    # 将 'None' 字符串转换为实际的 None
    data_matrix = np.where(data_matrix == 'None', None, data_matrix)
    
    # 计算 Krippendorff's Alpha
    print("正在计算 Krippendorff's Alpha...")
    alpha, coincidence_matrix, categories = krippendorff_alpha(data_matrix, level_of_measurement='nominal')
    
    print(f"\n{'='*80}")
    print(f"Krippendorff's Alpha = {alpha:.4f}")
    print(f"{'='*80}\n")
    
    # 分析不一致案例
    disagreements = analyze_disagreements(data_matrix, categories)
    
    print(f"不一致案例数量: {len(disagreements)} / {len(df)} ({len(disagreements)/len(df)*100:.2f}%)")
    print(f"完全一致案例数量: {len(df) - len(disagreements)} / {len(df)} ({(len(df)-len(disagreements))/len(df)*100:.2f}%)")
    print()
    
    # 显示前10个不一致案例
    if disagreements:
        print("不一致案例详情（前10个）:")
        print("-" * 80)
        for i, case in enumerate(disagreements[:10]):
            item_id = case['item_id']
            excerpt_id = df.iloc[item_id-1]['excerpt_id']
            codes = case['codes']
            print(f"{i+1}. {excerpt_id}: {codes}")
        
        if len(disagreements) > 10:
            print(f"... 还有 {len(disagreements)-10} 个不一致案例")
    
    print()
    
    # 统计每个编码者的编码分布
    print("\n" + "="*80)
    print("编码分布统计")
    print("="*80)
    
    for i, coder in enumerate(coder_columns):
        codes = df[coder].value_counts().sort_index()
        print(f"\n{coder}:")
        print(codes.head(10))
        if len(codes) > 10:
            print(f"... 还有 {len(codes)-10} 个类别")
    
    # Alpha值评估标准
    print("\n" + "="*80)
    print("Krippendorff's Alpha 评估标准")
    print("="*80)
    print("""
根据 Krippendorff (2004) 的建议:

α ≥ 0.800  : 可靠性高，数据可以用于得出确定性结论
0.667 ≤ α < 0.800 : 可靠性尚可，仅允许得出试探性结论
α < 0.667  : 可靠性不足，应该放弃或重新编码

其他参考标准:
α = 1.000  : 完全一致
α = 0.900+ : 优秀
α = 0.800-0.899 : 良好
α = 0.700-0.799 : 可接受
α = 0.600-0.699 : 有问题
α < 0.600  : 不可接受
α ≤ 0.000  : 一致性不如随机
    """)
    
    # 给出本次分析的评估
    print("="*80)
    print(f"本次分析结果评估 (α = {alpha:.4f}):")
    print("="*80)
    
    if alpha >= 0.800:
        assessment = "✓ 可靠性高 - 四位编码者之间的一致性很好，数据可以用于得出确定性结论。"
    elif alpha >= 0.667:
        assessment = "△ 可靠性尚可 - 四位编码者之间存在一定程度的不一致，建议仅用于得出试探性结论。"
    else:
        assessment = "✗ 可靠性不足 - 四位编码者之间的一致性较差，建议重新培训编码者或修改编码方案。"
    
    print(assessment)
    print()
    
    # 提供改进建议
    if alpha < 0.800:
        print("改进建议:")
        print("1. 组织编码者培训会议，讨论不一致的案例")
        print("2. 明确编码标准和规则，制作编码手册")
        print("3. 对不一致案例进行协商，达成共识")
        print("4. 考虑增加编码示例，特别是边界案例")
        print("5. 可以考虑剔除有问题的编码者或样本后重新计算")
    
    print()
    print("="*80)
    print("分析完成")
    print("="*80)
    
    # 保存详细结果
    results_summary = {
        "Krippendorff's Alpha": alpha,
        "样本总数": len(df),
        "完全一致案例数": len(df) - len(disagreements),
        "不一致案例数": len(disagreements),
        "一致性比例": f"{(len(df)-len(disagreements))/len(df)*100:.2f}%",
        "评估结果": assessment
    }
    
    # 保存到文件
    with open('/Users/majun/Documents/research/论文撰写/ LLM LoRA-ship collision/论文/投稿/12.29修改/补实验/一致性检查实验/krippendorff_alpha_results.txt', 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("Krippendorff's Alpha 一致性分析报告\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Krippendorff's Alpha = {alpha:.4f}\n\n")
        
        f.write("结果摘要:\n")
        for key, value in results_summary.items():
            f.write(f"  {key}: {value}\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("评估标准:\n")
        f.write("="*80 + "\n")
        f.write("α ≥ 0.800: 可靠性高\n")
        f.write("0.667 ≤ α < 0.800: 可靠性尚可\n")
        f.write("α < 0.667: 可靠性不足\n\n")
        
        if disagreements:
            f.write("\n" + "="*80 + "\n")
            f.write(f"不一致案例详情 (共 {len(disagreements)} 个):\n")
            f.write("="*80 + "\n")
            for i, case in enumerate(disagreements):
                item_id = case['item_id']
                excerpt_id = df.iloc[item_id-1]['excerpt_id']
                codes = case['codes']
                f.write(f"{i+1}. {excerpt_id}: {codes}\n")
    
    print(f"\n详细结果已保存到: /Users/majun/Documents/research/论文撰写/ LLM LoRA-ship collision/论文/投稿/12.29修改/补实验/一致性检查实验/krippendorff_alpha_results.txt")


if __name__ == "__main__":
    main()

Krippendorff's Alpha 一致性分析

数据集包含 314 个样本，4 个编码者
编码者: coder_A, coder_B, coder_C, coder_D

正在计算 Krippendorff's Alpha...

Krippendorff's Alpha = 0.8806

不一致案例数量: 56 / 314 (17.83%)
完全一致案例数量: 258 / 314 (82.17%)

不一致案例详情（前10个）:
--------------------------------------------------------------------------------
1. E001: ['IS07', 'IS07', 'IS07', 'IS11']
2. E004: ['IS01', 'IS02', 'IS01', 'IS01']
3. E008: ['IS15', 'IS01', 'IS01', 'IS01']
4. E010: ['IS02', 'IS12', 'IS12', 'IS03']
5. E016: ['IS15', 'IS07', 'IS15', 'IS15']
6. E019: ['IS08', 'IS18', 'IS08', 'IS06']
7. E023: ['IS06', 'IS05', 'IS06', 'IS06']
8. E024: ['IS14', 'IS13', 'IS13', 'IS13']
9. E026: ['IS09', 'IS09', 'IS18', 'IS09']
10. E027: ['IS13', 'IS06', 'IS13', 'IS23']
... 还有 46 个不一致案例


编码分布统计

coder_A:
coder_A
IS01    73
IS02     7
IS03     3
IS05    12
IS06    13
IS07    37
IS08    11
IS09    53
IS10     5
IS11     8
Name: count, dtype: int64
... 还有 10 个类别

coder_B:
coder_B
IS01    74
IS02     6
IS03     3
IS04     1
IS05    11
IS06    